<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/AI%20News%20Headline%20Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI News Headline Generator
This notebook demonstrates how to use pre-trained Transformer models to automatically generate news headlines.

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

# Initialize the text-generation pipeline
# We use 'gpt2' as a lightweight starter model
generator = pipeline('text-generation', model='gpt2')

def generate_headline(prompt):
    results = generator(prompt, max_length=30, num_return_sequences=3, truncation=True)
    return [res['generated_text'] for res in results]

# Example usage
topic = "Breaking: New breakthrough in Artificial Intelligence"
headlines = generate_headline(topic)

print(f"Topic: {topic}\n")
for i, h in enumerate(headlines):
    print(f"Generated Headline {i+1}: {h}")

# Fine-tuning the Model on a News Dataset
To adapt the model to specific news styles, we use the `datasets` library and the `Trainer` API.

In [5]:
!pip install datasets

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset

# 1. Load a sample news dataset
dataset = load_dataset("ag_news", split='train[:1000]')

# 2. Prepare the tokenizer and model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. Tokenize the data
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 4. Set up Training Arguments
training_args = TrainingArguments(
    output_dir="./gpt2-news",
    eval_strategy="no",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=1,
    per_device_train_batch_size=4,
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
)

# Start the training process immediately after initialization
print("Starting training...")
trainer.train()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


ValueError: Expected input batch_size (512) to match target batch_size (4).

In [4]:
# Training moved to previous cell to ensure trainer is defined.